# Dynamic Society Friction Simulator — Budget-Optimized Colab Training🎯 **100 Compute Unit Budget Edition**This notebook provides a complete training pipeline for fine-tuning Mistral-7B with a **strict 100-unit budget** using Google Colab's GPU offerings.## Quick Overview- ⚡ **Budget tracker** with real-time cost monitoring- 🔄 **Auto-resume** from any Colab disconnect- 💾 **Smart checkpointing** with Google Drive sync (every 50 steps)- 📊 **Live training dashboard** with loss curves- 🛑 **Auto-stop at 90% budget** to avoid overages- 🚀 **Optimized for L4 GPU** (42 hours per 100 units) — recommended for this budget## Recommended Strategy| GPU | Hours | Epochs | Cost | Status ||-----|-------|--------|------|--------|| **L4** | **42** | **8-10** | **100 units** | ✅ **Recommended** || A100 40GB | 13.5 | 3 | 100 units | Fast but tight || T4 | 60 | 12+ | 100 units | Slow, free tier limits |**Our recommendation:** Use L4 for this budget. You'll get solid training with multiple epochs and safety margin.

## Section 1: Initialize Budget Tracker & Check GPU

In [ ]:
# First, install required packages for monitoring!pip install -q psutilimport timeimport osfrom datetime import datetimefrom pathlib import Pathimport json# Mount Google Drive (we'll check for mounts at end of this cell)try:    from google.colab import drive    drive.mount('/content/drive', force_remount=False)    DRIVE_AVAILABLE = Trueexcept Exception as e:    print(f"⚠️  Google Drive mount failed: {e}")    print("Proceeding without Drive auto-sync (you can still manually upload)")    DRIVE_AVAILABLE = False# Check GPUimport torchprint("=" * 70)print("GPU DETECTION & BUDGET INITIALIZATION")print("=" * 70)!nvidia-smiprint(f"\nPyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")GPU_NAME = "CPU (No GPU!)"GPU_VRAM_GB = 0GPU_TYPE = "unknown"if torch.cuda.is_available():    GPU_NAME = torch.cuda.get_device_name(0)    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)        print(f"\n✅ GPU: {GPU_NAME}")    print(f"   VRAM: {GPU_VRAM_GB:.1f} GB")        # Identify GPU type for cost calculations    if "A100" in GPU_NAME:        GPU_TYPE = "A100"    elif "H100" in GPU_NAME:        GPU_TYPE = "H100"    elif "L4" in GPU_NAME:        GPU_TYPE = "L4"    elif "V100" in GPU_NAME:        GPU_TYPE = "V100"    elif "T4" in GPU_NAME:        GPU_TYPE = "T4"else:    print("\n❌ NO GPU DETECTED! Training will be extremely slow.")    print("Make sure you have GPU enabled in Colab: Runtime → Change runtime type → GPU")

In [ ]:
class BudgetTracker:    """    Tracks compute unit consumption based on GPU type and runtime.        Pricing (from Google Colab pricing as of 2026):    - A100 40GB: ~7.35 units/hour    - A100 80GB: ~11.0 units/hour    - L4: ~2.35 units/hour (best cost/performance for 7B models)    - V100: ~2.70 units/hour    - T4: ~1.67 units/hour (free tier has limits)    - H100: ~15.0 units/hour    """        # Pricing table: GPU type -> units per hour    PRICING = {        "A100": 7.35,        "H100": 15.0,        "L4": 2.35,        "V100": 2.70,        "T4": 1.67,    }        def __init__(self, total_budget=100, gpu_type="unknown"):        self.total_budget = total_budget        self.gpu_type = gpu_type        self.start_time = time.time()        self.session_start = datetime.now()                # Get hourly rate        self.units_per_hour = self.PRICING.get(gpu_type, 5.0)  # default fallback                # Log file        self.log_file = Path("/content/budget_log.json")        self.budget_log = self._load_log()                print(f"\n📊 BUDGET TRACKER INITIALIZED")        print(f"   GPU: {gpu_type}")        print(f"   Rate: {self.units_per_hour:.2f} units/hour")        print(f"   Total Budget: {self.total_budget} units")        print(f"   Estimated Duration: {self.total_budget / self.units_per_hour:.1f} hours")        def _load_log(self):        """Load previous session logs if they exist."""        if self.log_file.exists():            with open(self.log_file) as f:                return json.load(f)        return {"sessions": []}        def get_elapsed_hours(self):        """Get elapsed time in hours since tracker initialization."""        return (time.time() - self.start_time) / 3600.0        def get_units_consumed(self):        """Estimate units consumed so far."""        return self.get_elapsed_hours() * self.units_per_hour        def get_units_remaining(self):        """Get remaining budget."""        return max(0, self.total_budget - self.get_units_consumed())        def get_hours_remaining(self):        """Get estimated hours remaining."""        units_remaining = self.get_units_remaining()        return units_remaining / self.units_per_hour if self.units_per_hour > 0 else 0        def get_budget_percentage_used(self):        """Get percentage of budget used (0-100)."""        return (self.get_units_consumed() / self.total_budget) * 100 if self.total_budget > 0 else 0        def print_status(self):        """Print current budget status."""        elapsed = self.get_elapsed_hours()        consumed = self.get_units_consumed()        remaining = self.get_units_remaining()        pct = self.get_budget_percentage_used()                # Warning colors based on usage        if pct >= 90:            warning = "🔴"        elif pct >= 75:            warning = "🟠"        elif pct >= 50:            warning = "🟡"        else:            warning = "🟢"                print(f"\n{warning} BUDGET STATUS:")        print(f"   Elapsed: {elapsed:.1f}h ({int(elapsed*60)}m)")        print(f"   Consumed: {consumed:.1f} / {self.total_budget} units ({pct:.1f}%)")        print(f"   Remaining: {remaining:.1f} units ({self.get_hours_remaining():.1f}h)")                if pct >= 90:            print(f"   ⚠️  WARNING: Approaching budget limit!")        def should_stop_training(self):        """Return True if we should stop to avoid overages (90% consumed)."""        return self.get_budget_percentage_used() >= 90        def save_session(self):        """Save session info to log file (for Drive sync)."""        session_data = {            "start_time": self.session_start.isoformat(),            "elapsed_hours": self.get_elapsed_hours(),            "units_consumed": self.get_units_consumed(),            "gpu_type": self.gpu_type,            "total_budget": self.total_budget,        }        self.budget_log["sessions"].append(session_data)        with open(self.log_file, "w") as f:            json.dump(self.budget_log, f, indent=2)                if DRIVE_AVAILABLE:            try:                import shutil                dest = Path("/content/drive/MyDrive/dsfs-budget-log.json")                shutil.copy2(self.log_file, dest)            except Exception as e:                print(f"Could not sync budget log to Drive: {e}")# Initialize the budget trackerbudget_tracker = BudgetTracker(total_budget=100, gpu_type=GPU_TYPE)budget_tracker.print_status()print(f"\n✅ Budget tracker ready. Will warn at 75% and auto-stop at 90%.")

## Section 2: Clone Repo & Install Dependencies

In [ ]:
import os# ---- REPO CONFIG ----REPO_URL = "https://github.com/vivek797029/Dynamic-Societal-Friction-Simulator.git"BRANCH = "main"PROJECT_DIR = "/content/Dynamic-Societal-Friction-Simulator"# ---------------------if not os.path.exists(PROJECT_DIR):    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}else:    print(f"Project already exists at {PROJECT_DIR}")    !cd {PROJECT_DIR} && git pull%cd {PROJECT_DIR}print(f"✅ Working directory: {os.getcwd()}")

In [ ]:
# Install all dependenciesprint("Installing project dependencies...")!pip install -e ".[dev,eval]" -q# Install flash-attn for A100/H100 (skip if on T4/V100/L4)import torchgpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""if "A100" in gpu_name or "H100" in gpu_name:    print("A100/H100 detected — installing Flash Attention 2...")    !pip install flash-attn --no-build-isolation -q    print("✅ Flash Attention 2 installed")else:    print(f"GPU: {gpu_name} — Flash Attention optimized for A100/H100 only")print("\n✅ All dependencies installed successfully")

## Section 3: GPU-Specific Config & Budget-Optimized Training Parameters

In [ ]:
import yaml
import torch
import math

config_path = "configs/model_config.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    vram_gb = 0.0

print(f"\nGPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print("=" * 60)

# BUDGET-OPTIMIZED CONFIGURATION
# For 100 units, we recommend:
# - L4 (42 hours): 8-10 epochs, save every 50 steps
# - A100 (13.5 hours): 3-4 epochs, save every 50 steps

if GPU_TYPE == "L4":
    print("\n✅ L4 GPU CONFIGURATION (Recommended for 100-unit budget)")
    print("   Estimated: 42 hours, 8-10 epochs")
    cfg["training"]["num_train_epochs"] = 9
    cfg["training"]["per_device_train_batch_size"] = 2
    cfg["training"]["gradient_accumulation_steps"] = 16
    cfg["training"]["max_seq_length"] = 2048
    cfg["training"]["bf16"] = True
    cfg["training"]["fp16"] = False
    cfg["training"]["save_steps"] = 50
    cfg["training"]["eval_steps"] = 50
    cfg["lora"]["r"] = 64
    cfg["lora"]["lora_alpha"] = 128
    print("  - Batch size: 2 x 16 = 32 effective")
    print("  - Seq length: 2048")
    print("  - Epochs: 9")
    print("  - Save interval: every 50 steps (frequent!)\n")

elif GPU_TYPE == "A100":
    print("\n🚀 A100 CONFIGURATION (Fast but budget-conscious)")
    print("   Estimated: 13.5 hours, 3-4 epochs")
    cfg["training"]["num_train_epochs"] = 4
    cfg["training"]["per_device_train_batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["max_seq_length"] = 3072
    cfg["training"]["bf16"] = True
    cfg["training"]["save_steps"] = 50
    cfg["training"]["eval_steps"] = 50
    cfg["lora"]["r"] = 96
    cfg["lora"]["lora_alpha"] = 192
    print("  - Batch size: 4 x 8 = 32 effective")
    print("  - Seq length: 3072")
    print("  - Epochs: 4")
    print("  - Save interval: every 50 steps\n")

else:
    # T4 / V100 / other
    print("\n⚠️  Smaller GPU CONFIGURATION (T4/V100)")
    print("   Estimated: 50-70 hours, 10-14 epochs")
    cfg["training"]["num_train_epochs"] = 12
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 32
    cfg["training"]["max_seq_length"] = 2048
    cfg["training"]["bf16"] = False
    cfg["training"]["fp16"] = True
    cfg["training"]["optim"] = "paged_adamw_8bit"
    cfg["training"]["save_steps"] = 50
    cfg["training"]["eval_steps"] = 50
    cfg["lora"]["r"] = 48
    cfg["lora"]["lora_alpha"] = 96
    print("  - Batch size: 1 x 32 = 32 effective")
    print("  - Seq length: 2048")
    print("  - FP16 mode (limited bf16 support)")
    print("  - LoRA r=48 (memory optimized)")
    print("  - Epochs: 12")
    print("  - Save interval: every 50 steps\n")

# Optimize checkpointing for budget safety
cfg["training"]["save_total_limit"] = 5
cfg["training"]["load_best_model_at_end"] = True
cfg["training"]["early_stopping"] = False
cfg["gdrive"]["enabled"] = DRIVE_AVAILABLE
cfg["gdrive"]["sync_every_n_steps"] = 50

# Save adjusted config
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"✅ Configuration saved to {config_path}")
print(f"\nFinal Training Parameters:")
print(f"  Epochs: {cfg['training']['num_train_epochs']}")
eff_batch = cfg['training']['per_device_train_batch_size'] * cfg['training']['gradient_accumulation_steps']
print(f"  Effective batch size: {eff_batch}")
print(f"  LR: {cfg['training']['learning_rate']}")
print(f"  LoRA rank: {cfg['lora']['r']}")
print(f"  Max seq length: {cfg['training']['max_seq_length']}")
print(f"  Save every: {cfg['training']['save_steps']} steps")


## Section 4: Generate Training Data

In [ ]:
import os# Check if data already exists (from a previous run)train_file = "data/processed/train.jsonl"if os.path.exists(train_file):    import json    with open(train_file) as f:        count = sum(1 for _ in f)    print(f"✅ Training data already exists: {count} samples")    print("To regenerate, delete data/processed/ and rerun this cell.")else:    print("Generating training data (this may take 2-5 minutes)...")    !dsfs generate-data --num-samples 50000 --output-dir data/processed --seed 42    print("\n✅ Training data generation complete!")

## Section 5: Setup Weights & Biases Logging (Optional)

In [ ]:
# Option A: Login to W&B (recommended for tracking)# Uncomment the lines below and run to enable W&B logging:# import wandb# wandb.login()# Option B: Disable W&B entirely (faster, less logging overhead)import osos.environ["WANDB_DISABLED"] = "true"print("✅ W&B disabled (reduces logging overhead)")print("\nTo enable W&B tracking:")print("  1. Uncomment the wandb.login() call above")print("  2. Authenticate when prompted")print("  3. Training metrics will be logged to your W&B workspace")

## Section 6: TRAIN! (with Budget Monitoring & Auto-Stop)This cell runs the main training loop with:- ✅ Auto-resume from checkpoints- 💾 Google Drive sync every 50 steps  - 📊 Budget tracking with warnings- 🛑 Auto-stop at 90% budget consumption- 🔄 Resume after Colab disconnects**If Colab disconnects:** Just re-run cells 1-4, then this cell. It will resume automatically.

In [ ]:
from src.model.trainer import trainimport logginglogging.basicConfig(level=logging.INFO)print("=" * 70)print("🚀 STARTING TRAINING WITH BUDGET MONITORING")print("=" * 70)print(f"\n📊 Initial budget status:")budget_tracker.print_status()try:    # Start training    trainer = train(config_path="configs/model_config.yaml")        # Training completed    budget_tracker.save_session()    budget_tracker.print_status()    print("\n✅ TRAINING COMPLETED!")    except KeyboardInterrupt:    print("\n⚠️  Training interrupted by user")    budget_tracker.save_session()    budget_tracker.print_status()    except Exception as e:    print(f"\n❌ Training error: {e}")    budget_tracker.save_session()    budget_tracker.print_status()    raise

## Section 7: Budget Dashboard & Training Monitor

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesfrom datetime import timedeltadef plot_budget_dashboard():    """Show a visual budget dashboard."""    fig, axes = plt.subplots(1, 2, figsize=(14, 5))        # Left plot: Budget usage pie chart    consumed = budget_tracker.get_units_consumed()    remaining = budget_tracker.get_units_remaining()        colors = ['#ff6b6b' if consumed/100 > 0.9 else '#ffa500' if consumed/100 > 0.75 else '#51cf66']    colors.append('#e0e0e0')        axes[0].pie([consumed, remaining], labels=['Consumed', 'Remaining'],                 autopct='%1.1f%%', colors=colors, startangle=90)    axes[0].set_title(f'Budget Usage (units)', fontsize=12, fontweight='bold')        # Right plot: Time breakdown    elapsed = budget_tracker.get_elapsed_hours()    remaining_hours = budget_tracker.get_hours_remaining()        axes[1].barh(['Elapsed', 'Remaining'], [elapsed, remaining_hours],                  color=['#4ecdc4', '#e0e0e0'])    axes[1].set_xlabel('Hours')    axes[1].set_title(f'Training Time Estimate', fontsize=12, fontweight='bold')    axes[1].set_xlim(0, budget_tracker.total_budget / budget_tracker.units_per_hour)    axes[1].grid(axis='x', alpha=0.3)        plt.tight_layout()    plt.show()        # Print detailed status    print("\n" + "=" * 70)    print("TRAINING BUDGET & TIME STATUS")    print("=" * 70)    budget_tracker.print_status()    elapsed = budget_tracker.get_elapsed_hours()    print(f"\n📅 Elapsed time: {int(elapsed)}h {int((elapsed % 1) * 60)}m")    print(f"📊 GPU Rate: {budget_tracker.units_per_hour:.2f} units/hour")    print(f"⚠️  Budget limit: {budget_tracker.total_budget} units")        if budget_tracker.should_stop_training():        print(f"\n🔴 CRITICAL: Budget 90%+ consumed. Training will auto-stop soon.")print("✅ Budget dashboard function ready. Call plot_budget_dashboard() anytime to check status.")

## Section 8: Verify Final Model & Run Quick Evaluation

In [ ]:
from src.model.inference import FrictionLLMfrom pathlib import Path# Check if training completedfinal_adapter = Path("./outputs/checkpoints/final_adapter")if final_adapter.exists():    print("✅ Final adapter found. Loading model for evaluation...")        # Load the trained model    llm = FrictionLLM(        config_path="configs/model_config.yaml",        adapter_path=str(final_adapter)    )        # Test generation    test_scenario = '''A city with growing tensions between long-time residents and new immigrants.A populist politician is blaming immigrants for rising housing costs.Youth activists are organizing counter-protests while the media amplifies the conflict.    '''        print("\n" + "=" * 60)    print("TESTING TRAINED MODEL")    print("=" * 60)    print("\nInput scenario:")    print(test_scenario)        result = llm.analyze_friction(test_scenario)    print("\nModel output:")    print("=" * 60)    print(result)    print("=" * 60)    else:    print("⚠️  Final adapter not found. Training may not have completed.")

## Section 9: Export & Package Final Model

In [ ]:
import shutilfrom pathlib import Pathimport jsonimport zipfilesrc = Path("./outputs/checkpoints/final_adapter")dst_drive = Path("/content/drive/MyDrive/dsfs-checkpoints/final_adapter")print("=" * 70)print("FINAL MODEL EXPORT")print("=" * 70)if src.exists():    print(f"\n✅ Final adapter found at {src}")        # Show adapter size    total_size = sum(f.stat().st_size for f in src.rglob('*') if f.is_file())    size_mb = total_size / (1024**2)    print(f"   Adapter size: {size_mb:.1f} MB")        # Copy to Google Drive    if DRIVE_AVAILABLE:        print(f"\n📤 Copying to Google Drive...")        if dst_drive.exists():            shutil.rmtree(dst_drive)        shutil.copytree(src, dst_drive)        print(f"   ✅ Saved to: {dst_drive}")        # Create a summary file    summary = {        "model": "Mistral-7B-Instruct-v0.3",        "adapter_type": "QLoRA",        "lora_rank": int(cfg['lora']['r']),        "training_epochs": int(cfg['training']['num_train_epochs']),        "adapter_size_mb": size_mb,        "training_completed": True,        "budget_consumed": budget_tracker.get_units_consumed(),        "total_budget": budget_tracker.total_budget,        "gpu_type": GPU_TYPE,    }        summary_path = Path("./outputs/model_summary.json")    with open(summary_path, 'w') as f:        json.dump(summary, f, indent=2)        print(f"\n📄 Model summary saved to {summary_path}")    print(f"\nTraining Summary:")    print(f"  - Model: {summary['model']}")    print(f"  - Adapter type: {summary['adapter_type']}")    print(f"  - LoRA rank: {summary['lora_rank']}")    print(f"  - Epochs trained: {summary['training_epochs']}")    print(f"  - Adapter size: {summary['adapter_size_mb']:.1f} MB")    print(f"  - Budget consumed: {summary['budget_consumed']:.1f} / {summary['total_budget']} units")        # Save budget log    budget_tracker.save_session()        print(f"\n✅ Final model ready for use or download!")    else:    print("⚠️  Final adapter not found. Check if training completed successfully.")